
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" style="width: 600px">
</div>


# Lab: Adding Our Own Data to a Multi-Stage Reasoning System

### Working with external knowledge bases 
In this notebook we're going to augment the knowledge base of our LLM with additional data. We will split the notebook into two halves:
- First, we will walk through how to load in a relatively small, local text file using a `DocumentLoader`, split it into chunks, and store it in a vector database using `ChromaDB`.
- Second, you will get a chance to show what you've learned by building a larger system with the complete works of Shakespeare. 
----
### ![Dolly](https://files.training.databricks.com/images/llm/dolly_small.png) Learning Objectives

By the end of this notebook, you will be able to:
1. Add external local data to your LLM's knowledge base via a vector database.
2. Construct a Question-Answer(QA) LLMChain to "talk to your data."
3. Load external data sources from remote locations and store in a vector database.
4. Leverage different retrieval methods to search over your data. 

Fill in your credentials.

In [1]:
# TODO
# For many of the services that we'll using in the notebook, we'll need a HuggingFace API key so this cell will ask for it:
# HuggingFace Hub: https://huggingface.co/inference-api

# import os
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = "<FILL IN>"
from dotenv import load_dotenv
load_dotenv()

True

## Building a Personalized Document Oracle

In this notebook, we're going to build a special type of LLMChain that will enable us to ask questions of our data. We will be able to "speak to our data".

### Step 1 - Loading Documents into our Vector Store
For this system we'll leverage the [ChromaDB vector database](https://www.trychroma.com/) and load in some text we have on file. This file is of a hypothetical laptop being reviewed in both long form and with brief customer reviews. We'll use LangChain's `TextLoader` to load this data.

In [2]:
from langchain_community.document_loaders import TextLoader

# We have some fake laptop reviews that we can load in
laptop_reviews = TextLoader("fake_laptop_reviews.txt", encoding="utf8")
document = laptop_reviews.load()
display(document)

[Document(metadata={'source': 'fake_laptop_reviews.txt'}, page_content='"The NovusTech AlphaBook is a hidden gem in the world of technology. Its sleek design and powerful performance make it the perfect companion for any professional or student. I am impressed with its lightning-fast speed and smooth multitasking capabilities. This laptop has exceeded my expectations!"\n\n"I regret buying the NovusTech AlphaBook. The battery life is extremely poor, lasting barely a couple of hours even on the lowest brightness setting. The keyboard feels cheap and uncomfortable to type on, and the touchpad is unresponsive. Overall, this laptop is a major disappointment, and I wouldn\'t recommend it."\n\n"The NovusTech AlphaBook is an average device that gets the job done. It offers decent performance for everyday tasks like web browsing and word processing. However, the build quality leaves much to be desired. The display is adequate, but nothing extraordinary. It\'s an okay option for someone on a tig

### Step 2 - Chunking and Embeddings

Now that we have the data in document format, we will split data into chunks using a `CharacterTextSplitter` and embed this data using Hugging Face's embedding LLM to embed this data for our vector store.

In [3]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import tempfile

tmp_laptop_dir = tempfile.TemporaryDirectory()
tmp_shakespeare_dir = tempfile.TemporaryDirectory()

# First we split the data into manageable chunks to store as vectors. There isn't an exact way to do this, more chunks means more detailed context, but will increase the size of our vectorstore.
text_splitter = CharacterTextSplitter(chunk_size=250, chunk_overlap=10)
texts = text_splitter.split_documents(document)

# Now we'll create embeddings for our document so we can store it in a vector store and feed the data into an LLM. We'll use the sentence-transformers model for out embeddings. https://www.sbert.net/docs/pretrained_models.html#sentence-embedding-models/
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

# Finally we make our Index using chromadb and the embeddings LLM
chromadb_index = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=tmp_laptop_dir.name,
)

Created a chunk of size 299, which is longer than the specified 250
Created a chunk of size 316, which is longer than the specified 250
Created a chunk of size 314, which is longer than the specified 250
Created a chunk of size 300, which is longer than the specified 250
Created a chunk of size 272, which is longer than the specified 250
Created a chunk of size 259, which is longer than the specified 250
Created a chunk of size 277, which is longer than the specified 250


### Step 3 - Creating our Document QA LLM Chain
With our data now in vector form we need an LLM and a chain to take our queries and create tasks for our LLM to perform. 

In [4]:
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# We want to make this a retriever, so we need to convert our index.  This will create a wrapper around the functionality of our vector database so we can search for similar documents/chunks in the vectorstore and retrieve the results:
retriever = chromadb_index.as_retriever()

# This chain will be used to do QA on the document. We will need
# 1 - A LLM to do the language interpretation
# 2 - A vector database that can perform document retrieval
# 3 - Specification on how to deal with this data (more on this soon)

hf_llm = HuggingFacePipeline.from_model_id(
    model_id="google/flan-t5-large",
    task="text2text-generation",
    model_kwargs={
        "temperature": 0,
        "max_length": 128
    },
)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = PromptTemplate.from_template(
    """Use the following context to answer the question.
If the answer is not in the context, say you don't know.

Context:
{context}

Question:
{question}

Answer:"""
)

laptop_qa = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | hf_llm
    | StrOutputParser()
)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Device set to use cuda:0


### Step 4 - Talking to Our Data
Now we are ready to send prompts to our LLM and have it use our prompt, the access to our data, and read the information, process, and return with a response.

In [5]:
# Let's ask the chain about the product we have.
laptop_name = laptop_qa.invoke("What is the full name of the laptop?")
display(laptop_name)

'NovusTech AlphaBook'

In [6]:
# Now we'll ask the chain about the product.
laptop_features = laptop_qa.invoke("What are some of the laptop's features?")
display(laptop_features)

"Its compact size makes it highly portable, and the battery life is decent. The keyboard is comfortable to type on, and the touchpad is responsive. It's a reliable option for students or professionals on the go."

In [7]:
# Finally let's ask the chain about the reviews.
laptop_reviews = laptop_qa.invoke("What is the general sentiment of the reviews?")
display(laptop_reviews)

'mixed'

## Exercise: Working with larger documents
This document was relatively small. So let's see if we can work with something bigger. To show how well we can scale the vector database, let's load in a larger document. For this we'll get data from the [Gutenberg Project](https://www.gutenberg.org/) where thousands of free-to-access texts. We'll use the complete works of William Shakespeare.

Instead of a local text document, we'll download the complete works of Shakespeare using the `GutenbergLoader` that works with the Gutenberg project: https://www.gutenberg.org

In [8]:
from langchain_community.document_loaders import GutenbergLoader

loader = GutenbergLoader(
    "https://www.gutenberg.org/cache/epub/100/pg100.txt"
)  # Complete works of Shakespeare in a txt file

all_shakespeare_text = loader.load()

### Question 1

Now it's your turn! Based on what we did previously, fill in the missing parts below to build your own QA LLMChain.

In [9]:
# TODO
text_splitter = CharacterTextSplitter(chunk_size=1024, chunk_overlap=256) #hint try chunk sizes of 1024 and an overlap of 256 (this will take approx. 10mins with this model to build our vector database index)
texts = text_splitter.split_documents(all_shakespeare_text)

model_name = "sentence-transformers/all-MiniLM-L6-v2" #hint, try "sentence-transformers/all-MiniLM-L6-v2" as your model
embeddings = HuggingFaceEmbeddings(model_name=model_name)
chromadb_index = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=tmp_shakespeare_dir.name,
)

### Question 2

Let's see if we can do what we did with the laptop reviews. 

Think about what is likely to happen now. Will this command succeed? 

(***Hint: think about the maximum sequence length of a model***)

In [10]:
# TODO
# Let's start with the simplest method: "Stuff" which puts all of the data into the prompt and asks a question of it:
retriever = chromadb_index.as_retriever()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = PromptTemplate.from_template(
    """Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:"""
)

qa = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | hf_llm
    | StrOutputParser()
)

query = "What happens in the play Hamlet?"
# Run the query
query_results_hamlet = qa.invoke(query)
query_results_hamlet

Token indices sequence length is longer than the specified maximum sequence length for this model (1127 > 512). Running this sequence through the model will result in indexing errors


'Hamlet and certain Players enter a hall in the Castle.'

### Question 3

Now that we're working with larger documents, we should be mindful of the input sequence limitations that our LLM has. 

Chain Types for document loader:

- [`stuff`](https://docs.langchain.com/docs/components/chains/index_related_chains#stuffing) - Stuffing is the simplest method, whereby you simply stuff all the related data into the prompt as context to pass to the language model.
- [`map_reduce`](https://docs.langchain.com/docs/components/chains/index_related_chains#map-reduce) - This method involves running an initial prompt on each chunk of data (for summarization tasks, this could be a summary of that chunk; for question-answering tasks, it could be an answer based solely on that chunk).
- [`refine`](https://docs.langchain.com/docs/components/chains/index_related_chains#refine) - This method involves running an initial prompt on the first chunk of data, generating some output. For the remaining documents, that output is passed in, along with the next document, asking the LLM to refine the output based on the new document.
- [`map_rerank`](https://docs.langchain.com/docs/components/chains/index_related_chains#map-rerank) - This method involves running an initial prompt on each chunk of data, that not only tries to complete a task but also gives a score for how certain it is in its answer. The responses are then ranked according to this score, and the highest score is returned.
  * NOTE: For this exercise, `map_rerank` will [error](https://github.com/hwchase17/langchain/issues/3970).

In [11]:
# map_reduce
map_prompt = PromptTemplate.from_template(
    """Answer the question based only on this document chunk.

Document chunk:
{context}

Question:
{question}

Answer:"""
)

reduce_prompt = PromptTemplate.from_template(
    """Given the following candidate answers from different document chunks,
produce one final answer to the question.

Question:
{question}

Candidate answers:
{answers}

Final answer:"""
)

map_chain = map_prompt | hf_llm | StrOutputParser()
reduce_chain = reduce_prompt | hf_llm | StrOutputParser()

query = "Who is the main character in the Merchant of Venice?"

docs = retriever.invoke(query)

mapped_answers = [
    map_chain.invoke(
        {
            "context": doc.page_content,
            "question": query,
        }
    )
    for doc in docs
]

query_results_venice = reduce_chain.invoke(
    {
        "question": query,
        "answers": "\n\n".join(mapped_answers),
    }
)

query_results_venice

'ANTONIO Bassanio Antonio Antonio'

### Question 4


In [12]:
# TODO
# That's much better! Let's try another type

# refine
initial_prompt = PromptTemplate.from_template(
    """Answer the question based only on this document chunk.

Document chunk:
{context}

Question:
{question}

Initial answer:"""
)

refine_prompt = PromptTemplate.from_template(
    """We have an existing answer:

{existing_answer}

Refine it using the new document chunk below.
If the new chunk does not help, keep the answer unchanged.

New document chunk:
{context}

Question:
{question}

Refined answer:"""
)

initial_chain = initial_prompt | hf_llm | StrOutputParser()
refine_chain = refine_prompt | hf_llm | StrOutputParser()

query = "What happens to romeo and juliet?"

docs = retriever.invoke(query)

if not docs:
    query_results_romeo = "I don't know."
else:
    answer = initial_chain.invoke(
        {
            "context": docs[0].page_content,
            "question": query,
        }
    )

    for doc in docs[1:]:
        answer = refine_chain.invoke(
            {
                "existing_answer": answer,
                "context": doc.page_content,
                "question": query,
            }
        )

    query_results_romeo = answer

query_results_romeo

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


'The relevant information is: By God’s fair ordinance conjoin together, And let their heirs, God, if Thy will be so, Enrich the time to come with smoothed-faced peace, With smiling plenty, and fair prosperous days. Abate the edge of traitors, gracious Lord, That would reduce these bloody days again, And make poor England weep in streams of blood.'

In [13]:
import json
import re

# map_rerank
rerank_prompt = PromptTemplate.from_template(
    """Answer the question based only on this document chunk.

Return your result as JSON with exactly these keys:
- "answer": string
- "score": integer from 0 to 100

Document chunk:
{context}

Question:
{question}
"""
)

rerank_chain = rerank_prompt | hf_llm | StrOutputParser()

def parse_result(text: str):
    try:
        data = json.loads(text)
        return {
            "answer": str(data["answer"]),
            "score": int(data["score"]),
        }
    except Exception:
        score_match = re.search(r'"score"\s*:\s*(\d+)', text)
        answer_match = re.search(r'"answer"\s*:\s*"(.*?)"', text, re.DOTALL)
        return {
            "answer": answer_match.group(1) if answer_match else text,
            "score": int(score_match.group(1)) if score_match else 0,
        }

query = "Who is the main character in the Merchant of Venice?"

docs = retriever.invoke(query)

scored_answers = []
for doc in docs:
    raw = rerank_chain.invoke(
        {
            "context": doc.page_content,
            "question": query,
        }
    )
    parsed = parse_result(raw)
    scored_answers.append(parsed)

best_result = max(scored_answers, key=lambda x: x["score"]) if scored_answers else {
    "answer": "I don't know.",
    "score": 0,
}

query_results_romeo = best_result["answer"]
query_results_romeo

'answer: ANTONIO'

## Submit your Results (edX Verified Only)

To get credit for this lab, click the submit button in the top right to report the results. If you run into any issues, click `Run` -> `Clear state and run all`, and make sure all tests have passed before re-submitting. If you accidentally deleted any tests, take a look at the notebook's version history to recover them or reload the notebooks.

In [14]:
tmp_laptop_dir.cleanup()
tmp_shakespeare_dir.cleanup()

&copy; 2023 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/>
<a href="https://databricks.com/privacy-policy">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use">Terms of Use</a> | <a href="https://help.databricks.com/">Support</a>